# 11a. Causal Inference: Estimate Choice-Implied Session Thresholds

**Role of this notebook.** Build the first causal-inference dataset by estimating a free session-specific selectivity threshold from real watch-ratio behavior. The threshold is estimated for real adjacent sessions that have enough observations to make the session-level likelihood stable.

**Steps.**

1. Load the notebook-06 structural sample and fixed structural objects: `mu`, `sigma`, `phi`, category ids, user ids, and session beliefs.
2. Select adjacent real session pairs within the same user and burst, requiring both sessions to have at least 30 interactions.
3. Build the unique-session set appearing in at least one selected transition.
4. Reuse notebook-06 measurement-density code to compute `log_f0` and `log_f1` for every selected interaction.
5. Estimate one free `tau_choice` per selected session by maximizing the observed watch-ratio mixture likelihood.
6. Save a session-level tau table, a transition-level table, and diagnostics for later causal-inference notebooks.

The key distinction is:

| Threshold | Definition |
|---|---|
| `tau_model_old` | solved from the structural search-cost equation in notebook 06 |
| `tau_choice` | estimated freely from each session's observed watch-ratio likelihood |

This notebook does **not** run DML and does **not** re-estimate `mu`, `sigma`, `c`, or `phi`.


## Model Object Held Fixed

For an interaction in user `i`, session `t`, and video category `k`, notebook 06 provides fixed utility moments:

$$
U_{ik}\sim N(\widehat\mu_{ik},\widehat\sigma_{ik}^2).
$$

Given a candidate free threshold `tau`, the high-utility probability is:

$$
P(z_{itj}=1\mid \tau)=P(U_{ik}>\tau)
=1-\Phi\left(\frac{\tau-\widehat\mu_{ik}}{\widehat\sigma_{ik}}\right).
$$

The watch-ratio measurement model is also fixed from notebook 06:

$$
\log(WR_{itj})\mid z_{itj}=s
\sim N(m_s(x_{itj};\widehat\phi), v_s(x_{itj};\widehat\phi)),
\quad s\in\{0,1\}.
$$

For each candidate `tau`, the per-row observed likelihood is:

$$
L_{itj}(\tau)
= P(z_{itj}=1\mid\tau) f_1(\log WR_{itj}\mid x_{itj};\widehat\phi)
+ P(z_{itj}=0\mid\tau) f_0(\log WR_{itj}\mid x_{itj};\widehat\phi).
$$

The session-level free threshold is:

$$
\widehat\tau^{choice}_{it}
= \arg\max_\tau \sum_{j\in session(i,t)} \log L_{itj}(\tau).
$$

This objective uses observed behavior to infer session selectivity without imposing the search-cost optimality equation.


In [1]:
from pathlib import Path
import json
import math
import time

import numpy as np
import pandas as pd
from scipy import optimize, special

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

ROOT = Path('/Users/haozhangao/Desktop/RecSys Research')
OUT_DIR = ROOT / 'python_code_new' / 'outputs'

STRUCTURAL_INTERACTIONS_PATH = OUT_DIR / 'structural_interactions.parquet'
STRUCTURAL_SESSIONS_PATH = OUT_DIR / 'structural_sessions.parquet'
STRUCTURAL_THETA_PHI_PATH = OUT_DIR / 'structural_estimates_theta_phi.npz'
STRUCTURAL_ARRAYS_PATH = OUT_DIR / 'structural_estimation_arrays.npz'

TAU_SESSIONS_PATH = OUT_DIR / 'causal_tau_choice_sessions.parquet'
TAU_TRANSITIONS_PATH = OUT_DIR / 'causal_tau_choice_transitions.parquet'
DIAGNOSTICS_PATH = OUT_DIR / 'causal_tau_choice_diagnostics.json'

MIN_SESSION_INTERACTIONS = 30
OPT_XATOL = 1e-4
VAR_FLOOR = 1e-4
BOUND_SIGMA_MULT = 8.0
GLOBAL_TAU_BOUNDS = (-20.0, 20.0)

for path in [STRUCTURAL_INTERACTIONS_PATH, STRUCTURAL_SESSIONS_PATH, STRUCTURAL_THETA_PHI_PATH, STRUCTURAL_ARRAYS_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print('outputs will be saved under:', OUT_DIR)


outputs will be saved under: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs


## 1. Load Structural Sample and Fixed Parameters

This section loads the real notebook-06 structural sample. The interaction table supplies observed real `log_watch_ratio` and the exact measurement covariates used by notebook 06. The session table supplies stable `sess_idx`, old model-implied `tau_hat`, and session-level identifiers.

The fixed utility mean matrix is reconstructed from notebook-06 low-rank theta parameters:

$$
\widehat\mu = b_{cat} + U V^\top.
$$

Then the reference-category normalization is applied, matching notebook 06:

$$
\widehat\mu_{ik}\leftarrow \widehat\mu_{ik} - \widehat\mu_{i,k_{ref}}.
$$


In [2]:
interactions = pd.read_parquet(STRUCTURAL_INTERACTIONS_PATH)
sessions = pd.read_parquet(STRUCTURAL_SESSIONS_PATH)
theta_phi_npz = np.load(STRUCTURAL_THETA_PHI_PATH, allow_pickle=True)
arrays_npz = np.load(STRUCTURAL_ARRAYS_PATH, allow_pickle=True)

user_ids = theta_phi_npz['user_ids'].astype(np.int64)
cat_ids = theta_phi_npz['cat_ids'].astype(np.int64)
reference_cat_idx = int(theta_phi_npz['reference_cat_idx'][0])

uid_to_idx = {int(uid): idx for idx, uid in enumerate(user_ids)}
cat_to_idx = {int(cid): idx for idx, cid in enumerate(cat_ids)}

mu_by_user = theta_phi_npz['theta_b_cat'][None, :] + theta_phi_npz['theta_U'] @ theta_phi_npz['theta_V'].T
mu_by_user = mu_by_user - mu_by_user[:, [reference_cat_idx]]
sigmas_by_user = arrays_npz['sigmas_by_user'].astype(np.float64)
session_beliefs = arrays_npz['session_beliefs'].astype(np.float64)

phi = {
    'cat_ids': cat_ids.copy(),
    'cat_id_to_idx': cat_to_idx,
    'alpha_mu0': theta_phi_npz['phi_alpha_mu0'].astype(np.float64),
    'alpha_mu1': theta_phi_npz['phi_alpha_mu1'].astype(np.float64),
    'beta_mu0': theta_phi_npz['phi_beta_mu0'].astype(np.float64),
    'beta_mu1': theta_phi_npz['phi_beta_mu1'].astype(np.float64),
    'alpha_logvar0': theta_phi_npz['phi_alpha_logvar0'].astype(np.float64),
    'alpha_logvar1': theta_phi_npz['phi_alpha_logvar1'].astype(np.float64),
    'beta_logvar0': theta_phi_npz['phi_beta_logvar0'].astype(np.float64),
    'beta_logvar1': theta_phi_npz['phi_beta_logvar1'].astype(np.float64),
}

if mu_by_user.shape != sigmas_by_user.shape:
    raise ValueError(f'mu shape {mu_by_user.shape} does not match sigma shape {sigmas_by_user.shape}')
if len(sessions) != session_beliefs.shape[0]:
    raise ValueError('structural_sessions and session_beliefs have different row counts')

print('interactions:', f'{len(interactions):,}')
print('sessions:', f'{len(sessions):,}')
print('users:', len(user_ids))
print('categories:', len(cat_ids))
print('reference category id:', int(cat_ids[reference_cat_idx]))
print('reference mu abs max:', float(np.abs(mu_by_user[:, reference_cat_idx]).max()))


interactions: 4,325,560
sessions: 81,566
users: 1777
categories: 39
reference category id: -124
reference mu abs max: 0.0


## 2. Build Session-Level Controls and Adjacent Transitions

The causal transition unit is an adjacent session pair within the same user and burst:

$$
(user_i, burst_b, session_t) \rightarrow (user_i, burst_b, session_{t+1}).
$$

A pair is eligible only if both sessions have at least `MIN_SESSION_INTERACTIONS` observations. This keeps the free threshold estimate from being driven by very short sessions.

The unique-session table contains every session that appears as either the current or next session in at least one selected transition.


In [3]:
session_sizes = (
    interactions
    .groupby(['user_id', 'burst_id', 'session', 'sess_idx'], observed=True)
    .size()
    .rename('n_interactions')
    .reset_index()
)

session_controls = sessions.merge(
    session_sizes,
    on=['user_id', 'burst_id', 'session', 'sess_idx'],
    how='left',
    validate='one_to_one',
)
if session_controls['n_interactions'].isna().any():
    raise ValueError('some structural sessions have no interaction rows')
session_controls['n_interactions'] = session_controls['n_interactions'].astype(int)

# Attach belief columns and belief-implied utility moments for later causal notebooks.
belief_cols = [f'B_k_{idx:02d}' for idx in range(len(cat_ids))]
belief_frame = pd.DataFrame(session_beliefs, columns=belief_cols)
belief_frame['sess_idx'] = sessions['sess_idx'].to_numpy(dtype=np.int64)
session_controls = session_controls.merge(belief_frame, on='sess_idx', how='left', validate='one_to_one')

sess_user_idx = session_controls['user_idx'].to_numpy(dtype=np.int64)
belief_mat = session_controls[belief_cols].to_numpy(dtype=np.float64)
mu_sess = mu_by_user[sess_user_idx]
sigma_sess = sigmas_by_user[sess_user_idx]
session_controls['belief_utility_mean'] = np.sum(belief_mat * mu_sess, axis=1)
second_moment = np.sum(belief_mat * (sigma_sess ** 2 + mu_sess ** 2), axis=1)
session_controls['belief_utility_std'] = np.sqrt(np.maximum(second_moment - session_controls['belief_utility_mean'].to_numpy() ** 2, 0.0))

# Rename old model-implied tau for clarity.
session_controls = session_controls.rename(columns={'tau_hat': 'tau_model_old'})

ordered = session_controls.sort_values(['user_id', 'burst_id', 'session']).copy()
for col in ['session', 'sess_idx', 'n_interactions', 'tau_model_old', 'belief_entropy_hat', 'belief_utility_mean', 'belief_utility_std']:
    ordered[f'next_{col}'] = ordered.groupby(['user_id', 'burst_id'], sort=False)[col].shift(-1)

candidate_pairs = ordered[ordered['next_session'].notna()].copy()
candidate_pairs['is_consecutive'] = candidate_pairs['next_session'].astype(int).eq(candidate_pairs['session'].astype(int) + 1)

selected_pairs = candidate_pairs[
    candidate_pairs['is_consecutive']
    & candidate_pairs['n_interactions'].ge(MIN_SESSION_INTERACTIONS)
    & candidate_pairs['next_n_interactions'].ge(MIN_SESSION_INTERACTIONS)
].copy()

selected_session_keys = pd.concat([
    selected_pairs[['user_id', 'burst_id', 'session']].copy(),
    selected_pairs[['user_id', 'burst_id', 'next_session']].rename(columns={'next_session': 'session'}).copy(),
], ignore_index=True).drop_duplicates()

selected_sessions = session_controls.merge(
    selected_session_keys,
    on=['user_id', 'burst_id', 'session'],
    how='inner',
    validate='one_to_one',
).sort_values(['user_id', 'burst_id', 'session']).reset_index(drop=True)

print('real structural sessions:', f'{len(session_controls):,}')
print('same-burst adjacent candidate pairs:', f'{len(candidate_pairs):,}')
print('selected adjacent pairs:', f'{len(selected_pairs):,}')
print('unique sessions selected for tau:', f'{len(selected_sessions):,}')
display(session_controls['n_interactions'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame().T)


real structural sessions: 81,566
same-burst adjacent candidate pairs: 76,318
selected adjacent pairs: 20,295
unique sessions selected for tau: 28,367


,count,mean,std,min,50%,75%,90%,95%,99%,max
n_interactions,81566.0,53.03141,83.718793,1.0,24.0,63.0,133.0,204.0,406.35,3299.0


## 3. Reuse Notebook-06 Measurement Density

The functions below mirror the notebook-06 measurement convention:

| Feature block | Covariates |
|---|---|
| mean model | log video duration, relative session rank, squared relative rank, hour sine/cosine, weekend, low-activity flag, author-history flag, history watch ratio |
| log-variance model | log video duration, relative session rank, hour sine/cosine, weekend |

For each selected interaction, the notebook computes:

$$
\log f_0 = \log f(\log WR_{itj}\mid z=0,x_{itj};\widehat\phi),
$$

$$
\log f_1 = \log f(\log WR_{itj}\mid z=1,x_{itj};\widehat\phi).
$$

These values are fixed inputs to the one-dimensional `tau_choice` optimizer.


In [4]:
def build_watch_feature_blocks(interaction_df):
    required = [
        'i_video_duration', 'sess_rank', 'sess_len',
        'ctx_hour_sin', 'ctx_hour_cos', 'ctx_is_weekend',
        'u_is_lowactive_period',
        'hist_has_author_history', 'hist_ema_watchratio',
    ]
    missing = [col for col in required if col not in interaction_df.columns]
    if missing:
        raise KeyError(f'interaction_df missing watch-model columns: {missing}')

    duration = interaction_df['i_video_duration'].to_numpy(dtype=np.float32, copy=False)
    log_duration = np.log1p(np.maximum(duration, 0.0)).astype(np.float32)

    sess_rank = interaction_df['sess_rank'].to_numpy(dtype=np.float32, copy=False)
    sess_len = interaction_df['sess_len'].to_numpy(dtype=np.float32, copy=False)
    rel_rank = (sess_rank / np.maximum(sess_len, 1.0)).astype(np.float32)
    rel_rank_sq = (rel_rank * rel_rank).astype(np.float32)

    hour_sin = interaction_df['ctx_hour_sin'].to_numpy(dtype=np.float32, copy=False)
    hour_cos = interaction_df['ctx_hour_cos'].to_numpy(dtype=np.float32, copy=False)
    weekend = interaction_df['ctx_is_weekend'].to_numpy(dtype=np.float32, copy=False)
    low_activity = interaction_df['u_is_lowactive_period'].to_numpy(dtype=np.float32, copy=False)
    has_author_history = interaction_df['hist_has_author_history'].to_numpy(dtype=np.float32, copy=False)
    ema_watchratio = interaction_df['hist_ema_watchratio'].to_numpy(dtype=np.float32, copy=False)

    X_mean = np.vstack([
        log_duration,
        rel_rank,
        rel_rank_sq,
        hour_sin,
        hour_cos,
        weekend,
        low_activity,
        has_author_history,
        ema_watchratio,
    ]).T.astype(np.float32, copy=False)

    X_logvar = np.vstack([
        log_duration,
        rel_rank,
        hour_sin,
        hour_cos,
        weekend,
    ]).T.astype(np.float32, copy=False)

    return X_mean, X_logvar


def stable_log_variance_from_eta(eta, min_var=VAR_FLOOR):
    eta = np.asarray(eta, dtype=np.float64)
    return np.logaddexp(eta, np.log(float(min_var)))


def predict_watch_moments(interaction_df, phi, min_var=VAR_FLOOR):
    X_mean, X_logvar = build_watch_feature_blocks(interaction_df)
    cat_idx = interaction_df['cat_idx_int'].to_numpy(dtype=np.int64, copy=False)

    m0 = phi['alpha_mu0'][cat_idx] + X_mean @ phi['beta_mu0']
    m1 = phi['alpha_mu1'][cat_idx] + X_mean @ phi['beta_mu1']
    eta0 = phi['alpha_logvar0'][cat_idx] + X_logvar @ phi['beta_logvar0']
    eta1 = phi['alpha_logvar1'][cat_idx] + X_logvar @ phi['beta_logvar1']

    logv0 = stable_log_variance_from_eta(eta0, min_var=min_var)
    logv1 = stable_log_variance_from_eta(eta1, min_var=min_var)
    return m0, logv0, m1, logv1


def log_normal_pdf_from_logvar(x, mean, log_var):
    x = np.asarray(x, dtype=np.float64)
    mean = np.asarray(mean, dtype=np.float64)
    log_var = np.asarray(log_var, dtype=np.float64)
    return -0.5 * (np.log(2.0 * np.pi) + log_var + (x - mean) ** 2 * np.exp(-log_var))


## 4. Estimate Free Session Thresholds

For numerical stability, the optimizer evaluates the mixture likelihood in log space. Define:

$$
a_{itj}(\tau)=\frac{\tau-\widehat\mu_{ik(j)}}{\widehat\sigma_{ik(j)}}.
$$

Then:

$$
\log P(z=1\mid\tau)=\log\Phi(-a_{itj}(\tau)),
$$

$$
\log P(z=0\mid\tau)=\log\Phi(a_{itj}(\tau)).
$$

The row log likelihood is computed with stable log-sum-exp:

$$
\log L_{itj}(\tau)
= \log\left[
\exp\{\log P(z=1\mid\tau)+\log f_1\}
+
\exp\{\log P(z=0\mid\tau)+\log f_0\}
\right].
$$

Each session is optimized independently with a bounded scalar optimizer.


In [5]:
selected_sess_idx = set(selected_sessions['sess_idx'].astype(int).tolist())
selected_interactions = interactions[interactions['sess_idx'].isin(selected_sess_idx)].copy()
selected_interactions = selected_interactions.sort_values(['sess_idx', 'row_id']).reset_index(drop=True)

obs_user_idx = selected_interactions['user_idx_int'].to_numpy(dtype=np.int64, copy=False)
obs_cat_idx = selected_interactions['cat_idx_int'].to_numpy(dtype=np.int64, copy=False)
obs_log_wr = selected_interactions['log_watch_ratio'].to_numpy(dtype=np.float64, copy=False)

obs_mu = mu_by_user[obs_user_idx, obs_cat_idx].astype(np.float64)
obs_sigma = np.maximum(sigmas_by_user[obs_user_idx, obs_cat_idx].astype(np.float64), 1e-8)

m0, logv0, m1, logv1 = predict_watch_moments(selected_interactions, phi)
log_f0 = log_normal_pdf_from_logvar(obs_log_wr, m0, logv0)
log_f1 = log_normal_pdf_from_logvar(obs_log_wr, m1, logv1)

if not np.isfinite(log_f0).all() or not np.isfinite(log_f1).all():
    raise ValueError('non-finite measurement log-density values')

selected_interactions['mu_obs'] = obs_mu
selected_interactions['sigma_obs'] = obs_sigma
selected_interactions['log_f0'] = log_f0
selected_interactions['log_f1'] = log_f1

print('selected sessions:', f'{len(selected_sessions):,}')
print('selected interactions:', f'{len(selected_interactions):,}')
display(selected_interactions[['log_watch_ratio', 'mu_obs', 'sigma_obs', 'log_f0', 'log_f1']].describe().T)


selected sessions: 28,367
selected interactions: 3,202,486


,count,mean,std,min,25%,50%,75%,max
log_watch_ratio,3202486.0,-0.622516,1.377474,-6.907755,-0.944977,-0.275761,0.133348,5.443708
mu_obs,3202486.0,2.162110,1.802617,-11.540832,1.018237,2.268938,3.540407,9.528789
sigma_obs,3202486.0,1.024289,0.186471,0.053733,0.942714,1.030137,1.110531,3.810512
log_f0,3202486.0,-1.825279,0.706023,-47.434827,-1.834036,-1.716962,-1.609034,-0.918182
log_f1,3202486.0,-10.229666,45.602659,-2735.146603,-1.849520,-0.106878,0.255860,0.941504


In [6]:
def session_neg_loglik(tau, mu, sigma, log_f0, log_f1):
    a = (float(tau) - mu) / sigma
    log_p1 = special.log_ndtr(-a)  # log P(U > tau)
    log_p0 = special.log_ndtr(a)   # log P(U <= tau)
    log_mix = np.logaddexp(log_p1 + log_f1, log_p0 + log_f0)
    return -float(np.sum(log_mix))


def estimate_tau_for_group(g):
    mu = g['mu_obs'].to_numpy(dtype=np.float64, copy=False)
    sigma = g['sigma_obs'].to_numpy(dtype=np.float64, copy=False)
    lf0 = g['log_f0'].to_numpy(dtype=np.float64, copy=False)
    lf1 = g['log_f1'].to_numpy(dtype=np.float64, copy=False)

    lo = float(np.min(mu - BOUND_SIGMA_MULT * sigma))
    hi = float(np.max(mu + BOUND_SIGMA_MULT * sigma))
    lo = max(lo, GLOBAL_TAU_BOUNDS[0])
    hi = min(hi, GLOBAL_TAU_BOUNDS[1])
    if not np.isfinite(lo) or not np.isfinite(hi) or lo >= hi:
        lo, hi = GLOBAL_TAU_BOUNDS

    result = optimize.minimize_scalar(
        session_neg_loglik,
        args=(mu, sigma, lf0, lf1),
        bounds=(lo, hi),
        method='bounded',
        options={'xatol': OPT_XATOL},
    )

    tau_hat = float(result.x)
    neg_loglik = float(result.fun)
    objective = -neg_loglik
    success = bool(result.success and np.isfinite(tau_hat) and np.isfinite(neg_loglik))

    # Optional curvature-based standard error. It is undefined at boundary solutions.
    h = max(1e-3, 1e-3 * max(1.0, abs(tau_hat)))
    se = np.nan
    is_boundary = (tau_hat <= lo + 2 * h) or (tau_hat >= hi - 2 * h)
    if success and not is_boundary:
        f_left = session_neg_loglik(tau_hat - h, mu, sigma, lf0, lf1)
        f_mid = session_neg_loglik(tau_hat, mu, sigma, lf0, lf1)
        f_right = session_neg_loglik(tau_hat + h, mu, sigma, lf0, lf1)
        curvature = (f_left - 2.0 * f_mid + f_right) / (h ** 2)
        if np.isfinite(curvature) and curvature > 1e-12:
            se = float(np.sqrt(1.0 / curvature))

    return {
        'tau_choice': tau_hat,
        'tau_choice_success': success,
        'tau_choice_objective': objective,
        'tau_choice_neg_loglik': neg_loglik,
        'tau_choice_se': se,
        'tau_bound_lo': lo,
        'tau_bound_hi': hi,
        'tau_at_bound': bool(success and (tau_hat <= lo + OPT_XATOL * 2 or tau_hat >= hi - OPT_XATOL * 2)),
        'optimizer_nfev': int(getattr(result, 'nfev', -1)),
    }

start = time.time()
records = []
for idx, (sess_idx, g) in enumerate(selected_interactions.groupby('sess_idx', sort=False), start=1):
    rec = estimate_tau_for_group(g)
    rec['sess_idx'] = int(sess_idx)
    records.append(rec)
    if idx % 5000 == 0:
        print(f'estimated {idx:,} / {len(selected_sessions):,} sessions after {time.time() - start:.1f}s')

tau_estimates = pd.DataFrame(records)
print('finished tau estimation in seconds:', round(time.time() - start, 2))
display(tau_estimates.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)


estimated 5,000 / 28,367 sessions after 1.6s


estimated 10,000 / 28,367 sessions after 3.1s


estimated 15,000 / 28,367 sessions after 4.6s


estimated 20,000 / 28,367 sessions after 6.1s


estimated 25,000 / 28,367 sessions after 7.6s


finished tau estimation in seconds: 8.61


,count,mean,std,min,1%,5%,50%,95%,99%,max
tau_choice,28367.0,-0.731170,4.660568,-19.018354,-13.471903,-10.042770,0.948213,3.201845,9.171426,19.999954
tau_choice_objective,28367.0,-98.600307,153.423594,-2986.397641,-723.034224,-351.346463,-59.088593,17.364876,51.443973,409.897027
tau_choice_neg_loglik,28367.0,98.600307,153.423594,-409.897027,-51.443973,-17.364876,59.088593,351.346463,723.034224,2986.397641
tau_choice_se,21680.0,3756.063068,31644.009558,0.006234,0.079111,0.108362,0.250183,1.403663,134811.838494,955274.737996
tau_bound_lo,28367.0,-9.258291,1.958921,-20.000000,-15.601238,-12.851864,-8.938436,-6.716514,-5.723822,0.712387
tau_bound_hi,28367.0,12.685348,2.780451,-1.762669,7.428596,8.747482,12.420356,17.842187,20.000000,20.000000
optimizer_nfev,28367.0,19.851271,11.170614,9.000000,10.000000,11.000000,14.000000,42.000000,44.000000,47.000000
sess_idx,28367.0,40959.420489,23700.273807,0.000000,728.660000,3961.300000,41489.000000,77741.700000,80732.340000,81563.000000


## 5. Build Session and Transition Outputs

The session-level output contains one row per selected unique session. The transition-level output contains one row per eligible adjacent pair and joins the estimated thresholds for both sides of the transition:

$$
\Delta \tau^{choice}_{i,t}=\widehat\tau^{choice}_{i,t+1}-\widehat\tau^{choice}_{i,t}.
$$

This notebook does not define the final treatment variable. Later causal-inference notebooks can combine these tau outcomes with belief and impression-distribution changes.


In [7]:
tau_sessions = selected_sessions.merge(tau_estimates, on='sess_idx', how='left', validate='one_to_one')
if tau_sessions['tau_choice'].isna().any():
    raise ValueError('some selected sessions are missing tau estimates')

tau_sessions = tau_sessions.rename(columns={'user_idx': 'user_idx_int'})

# Keep a compact but useful session-level table.
session_output_cols = [
    'user_id', 'user_idx_int', 'burst_id', 'session', 'sess_idx', 'n_interactions',
    'tau_choice', 'tau_choice_success', 'tau_choice_objective', 'tau_choice_neg_loglik', 'tau_choice_se',
    'tau_bound_lo', 'tau_bound_hi', 'tau_at_bound', 'optimizer_nfev',
    'tau_model_old', 'cost_hat', 'belief_entropy_hat', 'belief_utility_mean', 'belief_utility_std',
] + belief_cols

tau_sessions_out = tau_sessions[session_output_cols].sort_values(['user_id', 'burst_id', 'session']).reset_index(drop=True)

left_cols = [
    'user_id', 'user_idx_int', 'burst_id', 'session', 'sess_idx', 'n_interactions',
    'tau_choice', 'tau_model_old', 'belief_entropy_hat', 'belief_utility_mean', 'belief_utility_std',
] + belief_cols
right_cols = [
    'user_id', 'burst_id', 'session', 'sess_idx', 'n_interactions',
    'tau_choice', 'tau_model_old', 'belief_entropy_hat', 'belief_utility_mean', 'belief_utility_std',
]

left = tau_sessions_out[left_cols].rename(columns={
    'session': 'session',
    'sess_idx': 'sess_idx_t',
    'n_interactions': 'n_interactions_t',
    'tau_choice': 'tau_t',
    'tau_model_old': 'tau_model_old_t',
    'belief_entropy_hat': 'belief_entropy_t',
    'belief_utility_mean': 'belief_utility_mean_t',
    'belief_utility_std': 'belief_utility_std_t',
})
left = left.rename(columns={c: f'{c}_t' for c in belief_cols})

right = tau_sessions_out[right_cols].rename(columns={
    'session': 'next_session',
    'sess_idx': 'sess_idx_t1',
    'n_interactions': 'n_interactions_t1',
    'tau_choice': 'tau_t1',
    'tau_model_old': 'tau_model_old_t1',
    'belief_entropy_hat': 'belief_entropy_t1',
    'belief_utility_mean': 'belief_utility_mean_t1',
    'belief_utility_std': 'belief_utility_std_t1',
})

base_transitions = selected_pairs[[
    'user_id', 'user_idx', 'burst_id', 'session', 'next_session', 'n_interactions', 'next_n_interactions', 'sess_idx', 'next_sess_idx'
]].rename(columns={
    'user_idx': 'user_idx_int',
    'n_interactions': 'n_interactions_t_from_pair',
    'next_n_interactions': 'n_interactions_t1_from_pair',
    'sess_idx': 'sess_idx_t_from_pair',
    'next_sess_idx': 'sess_idx_t1_from_pair',
}).copy()
base_transitions['next_session'] = base_transitions['next_session'].astype(int)
base_transitions['sess_idx_t1_from_pair'] = base_transitions['sess_idx_t1_from_pair'].astype(int)
base_transitions['n_interactions_t1_from_pair'] = base_transitions['n_interactions_t1_from_pair'].astype(int)

transitions = base_transitions.merge(
    left,
    on=['user_id', 'user_idx_int', 'burst_id', 'session'],
    how='left',
    validate='many_to_one',
).merge(
    right,
    on=['user_id', 'burst_id', 'next_session'],
    how='left',
    validate='many_to_one',
)

transitions['delta_tau'] = transitions['tau_t1'] - transitions['tau_t']
transitions['delta_tau_model_old'] = transitions['tau_model_old_t1'] - transitions['tau_model_old_t']
transitions['delta_belief_entropy'] = transitions['belief_entropy_t1'] - transitions['belief_entropy_t']
transitions['delta_belief_utility_mean'] = transitions['belief_utility_mean_t1'] - transitions['belief_utility_mean_t']
transitions['delta_belief_utility_std'] = transitions['belief_utility_std_t1'] - transitions['belief_utility_std_t']

# Prefer the clean names requested in the spec.
transitions = transitions.drop(columns=[
    'n_interactions_t_from_pair', 'n_interactions_t1_from_pair',
    'sess_idx_t_from_pair', 'sess_idx_t1_from_pair',
])

required_transition_cols = ['tau_t', 'tau_t1', 'delta_tau', 'n_interactions_t', 'n_interactions_t1']
if transitions[required_transition_cols].isna().any().any():
    raise ValueError('transition table has missing required tau or size columns')

print('session output rows:', f'{len(tau_sessions_out):,}')
print('transition output rows:', f'{len(transitions):,}')
display(tau_sessions_out[['n_interactions', 'tau_choice', 'tau_model_old', 'tau_choice_objective', 'tau_choice_neg_loglik', 'tau_choice_se']].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)
display(transitions[['n_interactions_t', 'n_interactions_t1', 'tau_t', 'tau_t1', 'delta_tau']].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)


session output rows: 28,367
transition output rows: 20,295


,count,mean,std,min,1%,5%,50%,95%,99%,max
n_interactions,28367.0,112.894772,108.968398,30.000000,30.000000,33.000000,76.000000,318.000000,542.000000,3299.000000
tau_choice,28367.0,-0.731170,4.660568,-19.018354,-13.471903,-10.042770,0.948213,3.201845,9.171426,19.999954
tau_model_old,28367.0,1.775620,1.152578,-7.781788,-1.030322,-0.017000,1.819457,3.344154,3.636048,7.521847
tau_choice_objective,28367.0,-98.600307,153.423594,-2986.397641,-723.034224,-351.346463,-59.088593,17.364876,51.443973,409.897027
tau_choice_neg_loglik,28367.0,98.600307,153.423594,-409.897027,-51.443973,-17.364876,59.088593,351.346463,723.034224,2986.397641
tau_choice_se,21680.0,3756.063068,31644.009558,0.006234,0.079111,0.108362,0.250183,1.403663,134811.838494,955274.737996


,count,mean,std,min,1%,5%,50%,95%,99%,max
n_interactions_t,20295.0,119.871348,113.256491,30.000000,30.000000,33.000000,81.000000,336.000000,572.000000,2239.000000
n_interactions_t1,20295.0,115.127963,112.582970,30.000000,30.000000,33.000000,77.000000,328.000000,552.120000,3299.000000
tau_t,20295.0,-0.775926,4.678964,-19.018354,-13.605785,-10.130514,0.939856,3.101484,9.108742,19.999954
tau_t1,20295.0,-0.759626,4.672751,-19.018354,-13.484535,-10.041187,0.952991,3.206204,9.170956,19.999947
delta_tau,20295.0,0.016300,4.871717,-25.989528,-13.345540,-9.509213,0.004760,9.610364,13.651976,25.261313


## 6. Save Outputs and Diagnostics

The saved files are:

| File | Contents |
|---|---|
| `causal_tau_choice_sessions.parquet` | one row per selected session with `tau_choice`, old model tau, beliefs, and session controls |
| `causal_tau_choice_transitions.parquet` | one row per eligible adjacent pair with `tau_t`, `tau_t1`, and `delta_tau` |
| `causal_tau_choice_diagnostics.json` | sample sizes, tau quantiles, optimizer success rate, and old-vs-new tau correlation |

These files are inputs for later causal-inference notebooks.


In [8]:
tau_sessions_out.to_parquet(TAU_SESSIONS_PATH, index=False)
transitions.to_parquet(TAU_TRANSITIONS_PATH, index=False)

old_tau_corr = tau_sessions_out[['tau_choice', 'tau_model_old']].corr(method='pearson').iloc[0, 1]
old_tau_spearman = tau_sessions_out[['tau_choice', 'tau_model_old']].corr(method='spearman').iloc[0, 1]

diag = {
    'min_session_interactions': MIN_SESSION_INTERACTIONS,
    'num_structural_users': int(len(user_ids)),
    'num_real_structural_interactions': int(len(interactions)),
    'num_real_structural_sessions': int(len(session_controls)),
    'num_same_burst_adjacent_pairs_before_filter': int(len(candidate_pairs)),
    'num_same_burst_consecutive_adjacent_pairs_before_filter': int(candidate_pairs['is_consecutive'].sum()),
    'num_adjacent_pairs_after_min_interaction_filter': int(len(selected_pairs)),
    'num_unique_sessions_used_for_tau': int(len(tau_sessions_out)),
    'tau_success_rate': float(tau_sessions_out['tau_choice_success'].mean()),
    'tau_at_bound_rate': float(tau_sessions_out['tau_at_bound'].mean()),
    'session_size_summary': tau_sessions_out['n_interactions'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_dict(),
    'tau_choice_summary': tau_sessions_out['tau_choice'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_dict(),
    'tau_choice_objective_summary': tau_sessions_out['tau_choice_objective'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_dict(),
    'tau_choice_se_summary': tau_sessions_out['tau_choice_se'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_dict(),
    'tau_choice_vs_tau_model_old_pearson': None if not np.isfinite(old_tau_corr) else float(old_tau_corr),
    'tau_choice_vs_tau_model_old_spearman': None if not np.isfinite(old_tau_spearman) else float(old_tau_spearman),
    'cat_ids': [int(x) for x in cat_ids],
    'belief_columns': belief_cols,
    'input_paths': {
        'structural_interactions': str(STRUCTURAL_INTERACTIONS_PATH),
        'structural_sessions': str(STRUCTURAL_SESSIONS_PATH),
        'structural_theta_phi': str(STRUCTURAL_THETA_PHI_PATH),
        'structural_arrays': str(STRUCTURAL_ARRAYS_PATH),
    },
    'output_paths': {
        'tau_sessions': str(TAU_SESSIONS_PATH),
        'tau_transitions': str(TAU_TRANSITIONS_PATH),
        'diagnostics': str(DIAGNOSTICS_PATH),
    },
}
DIAGNOSTICS_PATH.write_text(json.dumps(diag, indent=2, allow_nan=True))

print('saved:', TAU_SESSIONS_PATH)
print('saved:', TAU_TRANSITIONS_PATH)
print('saved:', DIAGNOSTICS_PATH)
print(json.dumps({
    'pairs': diag['num_adjacent_pairs_after_min_interaction_filter'],
    'unique_sessions': diag['num_unique_sessions_used_for_tau'],
    'success_rate': diag['tau_success_rate'],
    'tau_old_pearson': diag['tau_choice_vs_tau_model_old_pearson'],
    'tau_old_spearman': diag['tau_choice_vs_tau_model_old_spearman'],
}, indent=2))


saved: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_tau_choice_sessions.parquet
saved: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_tau_choice_transitions.parquet
saved: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/causal_tau_choice_diagnostics.json
{
  "pairs": 20295,
  "unique_sessions": 28367,
  "success_rate": 1.0,
  "tau_old_pearson": -0.3845488064639214,
  "tau_old_spearman": -0.10945815458800412
}
